# Importing data and library

In [21]:
# Core Data Manipulation
library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(purrr)
library(tibble)
library(forcats)

# Visualization and Tables
library(ggplot2)
library(RColorBrewer)
library(gt)

# Data import

In [22]:
geral_results <- read.csv("../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv")
print(geral_results)

    gene_name        cell_type     LogFC        Pvalue  abs_res
1       ROBO2 cell_Fibroblasts -93.12926  0.000000e+00 26.15408
2       ROBO2            total -93.12926  0.000000e+00 26.15408
3        ANK3  cell_Epithelial -45.00915  0.000000e+00 20.59934
4        ANK3 cell_Neutrophils -21.68529  0.000000e+00 20.59934
5        ANK3         cell_RBC -19.93120  3.157099e-47 20.59934
6        ANK3           cell_T -22.85591  0.000000e+00 20.59934
7        ANK3     cell_Unknown -35.28922 3.014743e-190 20.59934
8        ANK3            total -45.00915  0.000000e+00 20.59934
9        ANK3            total -21.68529  0.000000e+00 20.59934
10       ANK3            total -19.93120  3.157099e-47 20.59934
11       ANK3            total -22.85591  0.000000e+00 20.59934
12       ANK3            total -35.28922 3.014743e-190 20.59934
13      CDH12    cell_Neuronal  36.48661  1.339790e-58 61.07365
14     NKAIN2       cell_Glial  77.76291  3.166906e-02 43.70830
15     NKAIN2    cell_Neuronal 146.59276

# Bubble Plot

dbscan signal per tissues

In [23]:
# --- 1. Data Cleaning & Label Formatting ---
# Mapping internal cell type names to publication-quality English labels
cell_type_lookup <- c(
  "cell_T" = "T cells", "cell_B" = "B cells", "cell_SmoothMuscle" = "Smooth Muscle cells",
  "cell_RBC" = "Red Blood cells", "cell_Neutrophils" = "Neutrophils", "cell_Neuronal" = "Neuronal cells",
  "cell_Monocytes" = "Monocytes", "cell_Monoctyes" = "Monocytes", 
  "cell_Mast" = "Mast cells", "cell_Macrophages" = "Macrophages",
  "cell_Fibroblasts" = "Fibroblasts", "cell_Fibroblast" = "Fibroblasts",
  "cell_Epithelial" = "Epithelial cells", "cell_Endothelial" = "Endothelial cells",
  "cell_Glial" = "Glial cells", "cell_Brain" = "Brain cells"
)

df_bubble <- geral_results %>%
  filter(!is.na(STRs_ID), !is.na(LogFC)) %>%
  mutate(
    # Extract motif and size from STRs_ID (e.g., "AT:10")
    str_variant = str_extract(STRs_ID, "[^:]+:[^:]+$"),
    
    # Create the desired label: GENE (Motif:Size)
    gene_variant_label = paste0(gene_name, " (", str_variant, ")"),
    
    # Dynamic cleaning of cell types using the lookup table
    cell_type_clean = case_when(
      cell_type %in% names(cell_type_lookup) ~ cell_type_lookup[cell_type],
      TRUE ~ str_to_title(str_remove(cell_type, "cell_"))
    ),
    
    abs_res = abs(as.numeric(abs_res)),
    LogFC = as.numeric(LogFC)
  ) %>%
  filter(is.finite(LogFC)) %>%
  # Sort alphabetically by gene name to avoid a messy X-axis
  mutate(gene_variant_label = fct_reorder(gene_variant_label, gene_name))

# --- 2. Safety Checks for Color Scales ---
# Ensures the scale limits are valid even with a narrow data range
fill_limits <- range(df_bubble$LogFC, na.rm = TRUE)
if (fill_limits[1] == fill_limits[2]) fill_limits <- c(fill_limits[1]-0.1, fill_limits[2]+0.1)

# --- 3. Bubble Plot Creation ---
p_bubble <- ggplot(df_bubble, aes(x = gene_variant_label, y = cell_type_clean)) +
  # Geometry with transparency and thin strokes to reduce overlap perception
  geom_point(aes(size = abs_res, fill = LogFC), 
             shape = 21, color = "black", stroke = 0.3, alpha = 0.7) +
  
  # Using facet_grid with space="free_x" for proportional gene spacing per tissue
  facet_grid(. ~ source_tissue, scales = "free_x", space = "free_x") + 
  
  scale_fill_gradientn(
    colors = rev(RColorBrewer::brewer.pal(11, "RdBu")),
    limits = fill_limits,
    name = "Log2 FC"
  ) +
  
  # Adjusted range to prevent large bubbles from overcrowding neighbors
  scale_size_continuous(
    range = c(4, 15), 
    name = "STR Intensity\n(abs_res)"
  ) +
  
  # Expand Y-axis to provide more "breathing room" between cell type rows
  scale_y_discrete(expand = expansion(add = 0.4)) + 
  
  theme_minimal() +
  labs(
    title = "Impact of STR Outliers by Gene and Variant",
    subtitle = "Size: Intensity | Color: Log2 FC direction",
    x = "Gene & STR Variant (Motif:Size)", 
    y = "Cell Type"
  ) +
  theme(
    # Fine-tuning X-axis text appearance
    axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, 
                               size = 10, face = "bold.italic"),
    axis.text.y = element_text(face = "bold", size = 12, color = "black"),
    
    # Spacing between tissue blocks
    panel.spacing.x = unit(1.2, "lines"),
    
    # Aesthetic styling for panel headers
    strip.background = element_rect(fill = "grey92", color = "white"),
    strip.text = element_text(face = "bold", size = 10),
    
    # Soft gridlines for horizontal readability
    panel.grid.major.y = element_line(color = "grey95"),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    
    plot.margin = margin(15, 15, 15, 15)
  )

# --- 4. Export with Optimized Dimensions ---
# Increasing the height here prevents crowding of cell types in the final file
ggsave("results/STR_Impact_Final_Optimized.png", p_bubble, 
        width = 18, height = 11, dpi = 400, bg = "white")

Warning message:
“There were 3 warnings in `mutate()`.
The first warning was:
ℹ In argument: `gene_variant_label = fct_reorder(gene_variant_label,
  gene_name)`.
Caused by warning in `mean.default()`:
! argument is not numeric or logical: returning NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 2 remaining warnings.”


table visualization of dbscan per tissues

In [37]:
# --- 1. Data Processing ---
table_summary <- geral_results %>%
  filter(cell_type != "total") %>%
  mutate(
    across(c(Pvalue, LogFC, abs_res, allele1_est, allele2_est, depth), as.numeric),
    median_allele = (allele1_est + allele2_est) / 2,
    cell_type = case_when(
      cell_type == "cell_T" ~ "T cells",
      TRUE ~ str_remove(cell_type, "cell_") %>% str_replace_all("_", " ")
    ),
    group = str_to_title(group)
  ) %>%
  group_by(STRs_ID, gene_name, source_tissue, cell_type, group) %>%
  summarise(
    n_obs = n(),
    Pvalue = min(Pvalue, na.rm = TRUE),
    LogFC = mean(LogFC[is.finite(LogFC)], na.rm = TRUE),
    abs_res = mean(abs_res, na.rm = TRUE),
    allele2 = mean(allele2_est, na.rm = TRUE),
    allele2_min = min(allele2_est, na.rm = TRUE),
    allele2_max = max(allele2_est, na.rm = TRUE),
    depth = mean(depth, na.rm = TRUE),
    median_allele = mean(median_allele, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_wider(
    names_from = group,
    values_from = c(n_obs, abs_res, allele2, allele2_min, allele2_max, depth, median_allele),
    names_glue = "{.value}_{group}"
  ) %>%
  arrange(source_tissue, Pvalue)

# --- 2. GT Table Creation ---
gt_final_table <- table_summary %>%
  gt(groupname_col = "source_tissue") %>%
  tab_header(
    title = md("**STR Outlier Impact: Full Quantitative Analysis**"),
  ) %>%
  cols_label(
    STRs_ID = md("**STR ID**"), 
    gene_name = md("**Gene**"),
    cell_type = md("**Cell Type**"), 
    Pvalue = md("*p*-val"), 
    LogFC = md("**Log2 FC**"),
    n_obs_Case = md("**n**"), 
    abs_res_Case = "Abs. Res", 
    allele2_Case = "Allele 2",
    allele2_min_Case = "Allele 2 Min",
    allele2_max_Case = "Allele 2 Max",
    depth_Case = "Depth", 
    median_allele_Case = "Median Allele",
    n_obs_Control = md("**n**"), 
    abs_res_Control = "Abs. Res", 
    allele2_Control = "Allele 2",
    allele2_min_Control = "Allele 2 Min",
    allele2_max_Control = "Allele 2 Max",
    depth_Control = "Depth", 
    median_allele_Control = "Median Allele"
  ) %>%
  tab_spanner(label = md("**Case Group**"), columns = ends_with("_Case")) %>%
  tab_spanner(label = md("**Control Group**"), columns = ends_with("_Control")) %>%
  cols_align(align = "center", columns = everything()) %>%
  fmt_scientific(columns = Pvalue, decimals = 2) %>%
  fmt_number(columns = contains("abs_res") | contains("LogFC") | 
              contains("median_allele") | contains("allele2_") | 
              contains("depth"), decimals = 2) %>%
  fmt_integer(columns = contains("n_obs")) %>%
  fmt_missing(columns = everything(), missing_text = "-") %>%
  tab_options(
    row_group.background.color = "grey96",
    row_group.font.weight = "bold",
    column_labels.font.weight = "bold",
    table.font.size = px(10),
    table.width = pct(100)
  )

# --- 3. Save Output ---
if(!dir.exists("results")) dir.create("results")
gtsave(gt_final_table, "results/STR_Centered_Quantitative_Table.html")

 Integration of data filtred in STRs_filter step and scRNA-seq analysis

Function

In [25]:
# ==============================================================================
# MASTER FUNCTION: INTEGRATED STR & scRNA-seq ANALYSIS (CORRIGIDA)
# ==============================================================================
# Correcoes implementadas:
#   1. Parametro `delim` para escolher entre read_csv / read_csv2
#   2. Diagnostico de genes antes do merge (amostra dos nomes)
#   3. Resolucao de conflitos de nomes (.x/.y) apos inner_join
#   4. Referencia robusta a coluna de intensidade via .data[[ ]]
#   5. fct_reorder por intensity (evita warning de mean() em string)
#   6. Checagem de NA antes do ggplot
# ==============================================================================

library(tidyverse)
library(gt)

analyze_str_expression <- function(str_path,
                                   scrna_path = "../../6_scovid_data",
                                   output_prefix = "STR_Analysis",
                                   plot_title = "Impact of STR Outliers by Gene and Variant",
                                   color_palette = "RdBu",
                                   delim = c("csv2", "csv")) {

  delim <- match.arg(delim)

  cat(sprintf("\n--- STARTING ANALYSIS: %s ---\n", plot_title))

  # 1. Load STR Data (formato flexivel)
  if (!file.exists(str_path)) stop(paste("STR file not found at:", str_path))

  if (delim == "csv2") {
    df_str <- read_csv2(str_path, show_col_types = FALSE)
    cat(sprintf("  STR file loaded (csv2 format, ; delimiter): %d rows\n", nrow(df_str)))
  } else {
    df_str <- read_csv(str_path, show_col_types = FALSE)
    cat(sprintf("  STR file loaded (csv format, comma delimiter): %d rows\n", nrow(df_str)))
  }

  # 2. Load scRNA-seq Data (dir with GSEs or single file)
  if (dir.exists(scrna_path)) {
    gse_to_tissue <- c("GSE145926" = "lung", "GSE157344" = "lung", "GSE159812" = "brain")
    gse_folders <- list.dirs(scrna_path, recursive = FALSE)

    raw_scrna <- gse_folders %>%
      basename() %>%
      lapply(function(gse) {
        files <- list.files(file.path(scrna_path, gse), pattern = "\\.csv$", full.names = TRUE)
        files %>%
          lapply(function(f) {
            df <- read_csv(f, show_col_types = FALSE)
            cell_type <- f %>%
              basename() %>%
              tools::file_path_sans_ext() %>%
              stringr::str_remove(paste0("^", gse, "_"))
            df$cell_type <- cell_type
            df$GEO_ID <- gse
            df$source_tissue <- gse_to_tissue[gse]
            df
          }) %>%
          bind_rows()
      }) %>%
      bind_rows()

    cat(sprintf("  scRNA-seq loaded from %d GSE folders (%d rows)\n",
                length(gse_folders), nrow(raw_scrna)))
  } else {
    if (!file.exists(scrna_path)) stop(paste("scRNA-seq file not found at:", scrna_path))
    raw_scrna <- read_csv(scrna_path, show_col_types = FALSE)
    cat(sprintf("  scRNA-seq loaded: %d rows\n", nrow(raw_scrna)))
  }

  cat(sprintf("  STR rows: %d\n", nrow(df_str)))

  if (nrow(df_str) == 0 || nrow(raw_scrna) == 0) {
    cat("X Empty dataset. Aborting.\n")
    return(NULL)
  }

  # 3. Cell Type Lookup
  cell_type_lookup <- c(
    "cell_T" = "T cells", "cell_B" = "B cells", "cell_SmoothMuscle" = "Smooth Muscle cells",
    "cell_RBC" = "Red Blood cells", "cell_Neutrophils" = "Neutrophils", "cell_Neuronal" = "Neuronal cells",
    "cell_Monocytes" = "Monocytes", "cell_Monoctyes" = "Monocytes",
    "cell_Mast" = "Mast cells", "cell_Macrophages" = "Macrophages",
    "cell_Fibroblasts" = "Fibroblasts", "cell_Fibroblast" = "Fibroblasts",
    "cell_Epithelial" = "Epithelial cells", "cell_Endothelial" = "Endothelial cells",
    "cell_Glial" = "Glial cells", "cell_Brain" = "Brain cells"
  )

  # 4. Data Integration & Cleaning
  raw_scrna_clean <- raw_scrna %>%
    mutate(`Gene Symbol` = str_trim(str_to_upper(as.character(`Gene Symbol`))))

  df_str_clean <- df_str %>%
    mutate(gene_name = str_trim(str_to_upper(as.character(gene_name))))

  # Diagnostico: amostra dos genes de cada lado
  cat(sprintf("\n  [DIAG] Sample scRNA Gene Symbols (n=5): %s\n",
              paste(head(unique(raw_scrna_clean[["Gene Symbol"]]), 5), collapse = ", ")))
  cat(sprintf("  [DIAG] Sample STR gene_name values (n=5): %s\n",
              paste(head(unique(df_str_clean$gene_name), 5), collapse = ", ")))

  # Identifica colunas duplicadas (exceto a chave do join) e as renomeia
  join_key <- "gene_name"
  common_cols <- intersect(names(raw_scrna_clean), names(df_str_clean))
  common_cols <- setdiff(common_cols, join_key)

  for (col in common_cols) {
    old_name <- col
    new_name <- paste0(col, "_str")
    cat(sprintf("  [INFO] Renaming duplicate column '%s' -> '%s' in STR data\n", old_name, new_name))
    names(df_str_clean)[names(df_str_clean) == old_name] <- new_name
  }

  # Join
  df_integrated <- raw_scrna_clean %>%
    inner_join(df_str_clean, by = setNames("gene_name", "Gene Symbol"),
               relationship = "many-to-many") %>%
    rename(gene_name = `Gene Symbol`)

  cat(sprintf("  Merged rows after Join: %d\n", nrow(df_integrated)))

  if (nrow(df_integrated) == 0) {
    cat("X No gene matches found. Aborting.\n")
    cat("  Possiveis causas:\n")
    cat("  - Genes do STR ausentes no dataset scRNA-seq (tente delim = 'csv')\n")
    cat("  - Nomes de genes usam sistema diferente (ex. Ensembl IDs)\n")
    return(NULL)
  }

  # Determina coluna de intensidade (compativel com dplyr < 1.1.0)
  candidates <- c("abs_res", "abs_res_str", "abs_residual",
                  "outlier_residuals", "outlier_residuals_str")
  intensity_col <- NULL
  for (col in candidates) {
    if (col %in% names(df_integrated)) {
      intensity_col <- col
      break
    }
  }
  if (is.null(intensity_col)) {
    stop("No intensity column found (abs_res, abs_residual, or outlier_residuals)")
  }
  cat(sprintf("  Using intensity column: %s\n", intensity_col))

  # Determina nome real da coluna group (pode ter sufixo _str se houve conflito)
  group_col <- if ("group" %in% names(df_integrated)) "group" else "group_str"
  cat(sprintf("  Using group column: %s\n", group_col))

  # Clean
  df_clean <- df_integrated %>%
    mutate(
      source_tissue = if_else(
        is.na(source_tissue) | source_tissue == "",
        "Unknown Tissue",
        as.character(source_tissue)
      ),
      group = if_else(
        is.na(.data[[group_col]]),
        "Unknown Group",
        as.character(.data[[group_col]])
      ),
      str_variant = if_else(
        !is.na(STRs_ID),
        str_extract(STRs_ID, "[^:]+:[^:]+$"),
        "Unknown_Variant"
      ),
      gene_variant_label = paste0(gene_name, " (", str_variant, ")"),
      cell_type_clean = case_when(
        cell_type %in% names(cell_type_lookup) ~
          unname(cell_type_lookup[cell_type]),
        is.na(cell_type) | cell_type == "" ~ "Unknown Cell Type",
        TRUE ~ str_to_title(str_remove(as.character(cell_type), "cell_"))
      ),
      LogFC = as.numeric(str_replace_all(as.character(LogFC), ",", ".")),
      Pvalue = as.numeric(str_replace_all(as.character(Pvalue), ",", ".")),
      intensity = abs(as.numeric(.data[[intensity_col]]))
    ) %>%
    mutate(LogFC = if_else(is.finite(LogFC), LogFC, 0))

  # Ordena levels alfabeticamente por gene_variant_label
  df_clean <- df_clean %>%
    mutate(gene_variant_label = factor(gene_variant_label,
                                       levels = sort(unique(gene_variant_label))))

  # Checagem de NA
  na_intensity <- sum(is.na(df_clean$intensity))
  na_logfc <- sum(is.na(df_clean$LogFC) | !is.finite(df_clean$LogFC))
  if (na_intensity > 0 || na_logfc > 0) {
    cat(sprintf("\n  [WARN] NAs encontrados - intensity: %d, LogFC: %d (de %d rows)\n",
                na_intensity, na_logfc, nrow(df_clean)))
  }

  # 5. Color Scale Limits
  fill_limits <- range(df_clean$LogFC, na.rm = TRUE)
  if (fill_limits[1] == fill_limits[2]) {
    fill_limits <- c(fill_limits[1] - 0.1, fill_limits[2] + 0.1)
  }

  # 6. Bubble Plot
  cat("[PROCESS] Generating Bubble Plot...\n")
  p <- ggplot(df_clean, aes(x = gene_variant_label, y = cell_type_clean)) +
    geom_point(aes(size = intensity, fill = LogFC),
               shape = 21, color = "black", stroke = 0.3, alpha = 0.7) +
    facet_grid(. ~ source_tissue, scales = "free_x", space = "free_x") +
    scale_fill_gradientn(
      colors = rev(RColorBrewer::brewer.pal(11, color_palette)),
      limits = fill_limits,
      name = "Log2 FC"
    ) +
    scale_size_continuous(range = c(4, 15), name = "STR Intensity") +
    scale_y_discrete(expand = expansion(add = 0.4)) +
    theme_minimal() +
    labs(
      title = plot_title,
      subtitle = "Size: Intensity | Color: Log2 FC direction",
      x = "Gene & STR Variant (Motif:Size)",
      y = "Cell Type"
    ) +
    theme(
      axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1,
                                  size = 10, face = "bold.italic"),
      axis.text.y = element_text(face = "bold", size = 12, color = "black"),
      panel.spacing.x = unit(1.2, "lines"),
      strip.background = element_rect(fill = "grey92", color = "white"),
      strip.text = element_text(face = "bold", size = 10),
      panel.grid.major.y = element_line(color = "grey95"),
      panel.grid.major.x = element_blank(),
      plot.margin = margin(15, 15, 15, 15)
    )

  # 7. Quantitative Summary Table
  cat("[PROCESS] Generating Quantitative Table...\n")

  # Determina o nome real da coluna group (pode ter sufixo _str se houve conflito)
  group_col <- if ("group" %in% names(df_clean)) "group" else "group_str"

  table_data <- df_clean %>%
    mutate(
      across(any_of(c("Pvalue", "LogFC", "intensity", "allele1_est",
                       "allele2_est", "depth")), as.numeric),
      median_allele = if (all(c("allele1_est", "allele2_est") %in% colnames(.))) {
        (.data[["allele1_est"]] + .data[["allele2_est"]]) / 2
      } else {
        NA_real_
      },
      group_clean = str_to_title(.data[[group_col]])
    ) %>%
    group_by(STRs_ID, gene_name, source_tissue, cell_type_clean, group_clean) %>%
    summarise(
      n_obs = n(),
      Pvalue = min(Pvalue, na.rm = TRUE),
      LogFC = mean(LogFC, na.rm = TRUE),
      abs_res = mean(intensity, na.rm = TRUE),
      allele2 = if ("allele2_est" %in% colnames(.)) mean(allele2_est, na.rm = TRUE) else NA_real_,
      depth = if ("depth" %in% colnames(.)) mean(depth, na.rm = TRUE) else NA_real_,
      median_allele = mean(median_allele, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    rename(group = group_clean) %>%
    pivot_wider(
      names_from = group,
      values_from = c(n_obs, abs_res, allele2, depth, median_allele),
      names_glue = "{.value}_{group}"
    )

  gt_table <- table_data %>%
    gt(groupname_col = "source_tissue") %>%
    tab_header(title = md(paste0("**Quantitative Analysis:** ", plot_title))) %>%
    cols_label(
      STRs_ID = md("**STR ID**"),
      gene_name = md("**Gene**"),
      cell_type_clean = md("**Cell Type**"),
      Pvalue = md("*p*-val"),
      LogFC = md("**Log2 FC**")
    ) %>%
    tab_spanner(label = md("**Case Group**"), columns = contains("_Case")) %>%
    tab_spanner(label = md("**Control Group**"), columns = contains("_Control")) %>%
    cols_align(align = "center", columns = everything()) %>%
    fmt_scientific(columns = Pvalue, decimals = 2) %>%
    fmt_number(columns = contains("abs_res") | contains("LogFC") |
                contains("median_allele"), decimals = 2) %>%
    fmt_missing(columns = everything(), missing_text = "-") %>%
    tab_options(
      row_group.background.color = "grey96",
      row_group.font.weight = "bold",
      table.font.size = px(10)
    )

  # 8. Export
  if (!dir.exists("results")) dir.create("results")
  ggsave(sprintf("results/%s_Plot_Optimized.png", output_prefix), p,
         width = 18, height = 11, dpi = 400, bg = "white")
  gtsave(gt_table, sprintf("results/%s_Table.html", output_prefix))

  cat("\n[EXPORT] Saving GEO-aware CSV...\n")
  if ("GEO_ID" %in% colnames(df_clean)) {
    df_clean %>%
      relocate(GEO_ID, source_tissue) %>%
      write_csv(sprintf("results/%s_GEO_Data.csv", output_prefix))
    cat(sprintf("  -> results/%s_GEO_Data.csv\n", output_prefix))
  } else {
    cat("  -> [WARNING] GEO_ID not found, saving without it.\n")
    df_clean %>%
      relocate(any_of(c("source_tissue"))) %>%
      write_csv(sprintf("results/%s_Data.csv", output_prefix))
  }

  cat(sprintf("\n✓ Process completed for prefix: %s\n", output_prefix))
  cat(sprintf("  Plot  : results/%s_Plot_Optimized.png\n", output_prefix))
  cat(sprintf("  Table : results/%s_Table.html\n", output_prefix))

  return(list(plot = p, table = gt_table))
}

Top 10 Outliers

In [26]:
res1 <- analyze_str_expression(
  str_path = "../7.3_STRs_filter/results/top_10_unique_loci.csv",
  output_prefix = "Top10_Outliers",
  plot_title = "Top 10 STR Outliers by Residual Magnitude"
)


--- STARTING ANALYSIS: Top 10 STR Outliers by Residual Magnitude ---


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.



  STR file loaded (csv2 format, ; delimiter): 434 rows
  scRNA-seq loaded from 2 GSE folders (1460 rows)
  STR rows: 434

  [DIAG] Sample scRNA Gene Symbols (n=5): ACTB, AFF3, ANKRD28, AP001011.1, B2M
  [DIAG] Sample STR gene_name values (n=5): HMGCLL1, ., CDK19, ITFG2-AS1, TAFA4
  Merged rows after Join: 0
X No gene matches found. Aborting.
  Possiveis causas:
  - Genes do STR ausentes no dataset scRNA-seq (tente delim = 'csv')
  - Nomes de genes usam sistema diferente (ex. Ensembl IDs)


Major alelle without overlap

In [27]:
res2 <- analyze_str_expression(
  str_path = "../7.3_STRs_filter/results/covid_targeted_no_overlap_allele2.csv",
  output_prefix = "overlap_allele2",
  plot_title = "STRs without overlap in the major allele between groups"
)


--- STARTING ANALYSIS: STRs without overlap in the major allele between groups ---


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.



  STR file loaded (csv2 format, ; delimiter): 0 rows
  scRNA-seq loaded from 2 GSE folders (1460 rows)
  STR rows: 0
X Empty dataset. Aborting.


Mean alelle without overlap

In [28]:
res_a2 <- analyze_str_expression(
  str_path        = "../7.3_STRs_filter/results/covid_targeted_no_overlap_mean_allele.csv",
  output_prefix   = "overlap_mean_allele",
  plot_title      = "STRs without overlap in the mean allele between groups",
  color_palette   = "RdBu" 
)


--- STARTING ANALYSIS: STRs without overlap in the mean allele between groups ---


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.



  STR file loaded (csv2 format, ; delimiter): 0 rows
  scRNA-seq loaded from 2 GSE folders (1460 rows)
  STR rows: 0
X Empty dataset. Aborting.
